# Module 3 — LangChain & RAG: Topics & Practice

## Learning outcomes
1. Compose chains with LangChain's LCEL syntax.
2. Build a RAG pipeline end-to-end and identify each stage's failure modes.
3. Pick a vector DB on real criteria, not vibes.
4. Apply *advanced* retrieval (hybrid, rerank, multi-query) and *measure* the lift.

## 1. LangChain in 5 minutes

LCEL = pipe-able runnables. The mental model:
```
prompt | llm | parser
```
Each step is a `Runnable`; you can `.invoke`, `.batch`, `.stream`, or `.ainvoke`. Memory and callbacks are wrappers around runnables, not separate worlds.

> **Exercise 3.1** — Build a runnable chain: takes `{question}` → produces a summarised answer. Add a JSON parser. No RAG yet.

## 2. The RAG pipeline — every stage breaks differently

```
docs → split → embed → index → (query → embed → retrieve → rerank) → prompt → llm
```

| Stage | Most common failure | Mitigation |
|---|---|---|
| Split | Cut mid-sentence, lose context | Recursive splitter, semantic splitter |
| Embed | Domain mismatch | Domain-tuned embeddings, hybrid w/ BM25 |
| Index | Wrong distance metric | cosine for normalized, dot for raw |
| Retrieve | Top-k returns near-duplicates | MMR, dedup |
| Generate | Hallucinates outside context | Strict prompt + citation requirement + refusal |

> **Exercise 3.2** — Take a single 100-page PDF. Run it through three chunk sizes (256/512/1024 tokens) and report Recall@5 on 10 hand-written questions.

## 3. Vector DBs — the choice that matters less than you think

| DB | Pick when |
|---|---|
| FAISS | local, no server, <10M vectors |
| Chroma | local, simple, <1M |
| Qdrant | self-host, great filters, hybrid built-in |
| Pinecone | managed, pay to skip ops |
| Weaviate | hybrid + multi-modal first-class |
| pgvector | you already have Postgres |

**Decision driver:** filtering needs and ops budget, *not* benchmark TPS at scale you don't have.

> **Exercise 3.3** — For your capstone corpus, sketch the metadata schema (source, page, author, date). Which DB lets you filter on `date >= 2024 AND source IN (...)` cleanly?

## 4. Embeddings — pick deliberately

- `text-embedding-3-small` (OpenAI) — strong default, cheap.
- `voyage-3` — top of MTEB for retrieval.
- `bge-large-en-v1.5` — best free / self-host.
- `e5-mistral-7b` — heavy but excellent.
- **Domain fine-tunes** worth it when generic embeddings sit at <60% Recall@5.

> **Exercise 3.4** — Embed 100 of your domain documents with two different models. Cluster (UMAP + KMeans). Eyeball: which model separates concepts more cleanly?

## 5. Advanced retrieval — what to add when basic RAG plateaus

1. **Hybrid (BM25 + dense)** — recovers exact-match queries dense models miss.
2. **Multi-query** — LLM rewrites the query into 3–5 variants, union the results.
3. **HyDE** — generate a *hypothetical* answer, embed *that* for retrieval.
4. **Re-ranking** — Cohere `rerank-3` or `bge-reranker-large` over top-50, keep top-5.
5. **Parent-document** — embed small chunks for retrieval, return the larger parent for context.
6. **Self-RAG / corrective RAG** — model decides when to retrieve and critiques retrieved snippets.

> **Exercise 3.5** — Take your basic RAG. Add re-ranking only. Measure faithfulness lift on a 30-question eval. Was it worth the latency?